# 14 How much is in the input?

Where the depth-5 models lose their score, and whether more model could win it back. Inference only, for the real-space model (`subhalo_C_mixed_smooth2ch_depth5`) and the redshift-space one (`..._rsd`), at λ = 3, 5, 7 and 10.

Distance here is from each voxel to the nearest non-empty cell of the input grid (the nearest tracer), in h⁻¹ Mpc.

1. **What is far from every tracer**: per true class, the share of test-slab voxels in each distance bin.
2. **Model against a distance-only classifier**: accuracy against distance for the model and for a classifier that sees only the distance (the most common class in each 0.5 h⁻¹ Mpc bin, fitted on the validation slab). Where the two match, the model gets nothing out of the input beyond how far the nearest tracer is.
3. **How much score sits far from tracers**: the four-class MCC if every voxel farther than D from a tracer were classified correctly, against D. Not an achievable score; it shows how much of the loss is in voxels the input says little about.
4. **Training against test slab**: four-class MCC on the training, validation and test slabs. A large gap means the model fits what it has seen better than new regions (more data could help); no gap means more data or capacity is less likely to help.

Runtime: **L4 GPU** with **High-RAM**. About 20–30 minutes (8 whole-box predictions plus distance transforms).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

In [ ]:
import gc
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from deepvoid import Catalog
from deepvoid import interpret as I
from deepvoid.config import load_config
from deepvoid.evaluate import confusion_matrix, scores
from deepvoid.inputs import build
from deepvoid.labels import CLASS_NAMES
from deepvoid.models import load_model
from deepvoid.predict import predict_volume
from deepvoid.report import Report
from deepvoid.split import Region

root = Path('/content/drive/MyDrive/deepvoid_runs')
MODELS = {                                   # label: (run folder, input name pattern)
    'real space': ('subhalo_C_mixed_smooth2ch_depth5', 'tng300-3/subhalos/L{L}'),
    'redshift space': ('subhalo_C_mixed_smooth2ch_depth5_rsd', 'tng300-3/subhalos/L{L}/rsd-z'),
}
SPACINGS = [3, 5, 7, 10]
FINE = np.arange(0, 16.5, 0.5)               # distance bins for the distance-only classifier [h^-1 Mpc]
COARSE = np.array([0, 2, 4, 6, 8, 1e9])      # bins for the class shares
COARSE_NAMES = ['0–2', '2–4', '4–6', '6–8', '> 8']
CUTOFFS = [2, 4, 6, 8, 10, 12]               # for "MCC if fixed beyond D"
PAD = 40                                     # cells; distances up to 40 cells (16 h^-1 Mpc) are exact
catalog = Catalog()
rep = Report(root / 'information_limit')
print('saving to', rep.folder)


def load_run(name):
    out = root / name
    cfg = load_config(out / 'config.yaml')
    regions = {k: Region(**v) for k, v in json.loads((out / 'split.json').read_text()).items()}
    norms = json.loads((out / 'normalization.json').read_text())
    return cfg, regions, norms, load_model(out / 'best.keras')


def mcc(truth, pred):
    return scores(confusion_matrix(truth.ravel(), pred.ravel()))['mcc']

## Predictions and distances

One whole-box prediction per model and spacing, stitched the way each run was scored. Only the class maps and the distances on the validation and test slabs are kept.

In [ ]:
res = {}
for label, (run, pattern) in MODELS.items():
    cfg, regions, norms, model = load_run(run)
    truth = catalog.load(cfg['data']['labels']).data
    n = truth.shape[0]
    train = regions['train'].mask(n)
    ev = cfg['evaluate']
    res[label] = {}
    for L in SPACINGS:
        name = pattern.format(L=L)
        vol = catalog.load(name)
        cell = vol.box_size / n
        field = build(catalog, name, cfg['data']['smoothing'], norms, train)
        probs = predict_volume(model.predict_on_batch, field, crop=ev['crop'], margin=ev['margin'],
                               batch_size=ev['batch_size'])
        pred = probs.argmax(-1).astype(np.uint8)
        del probs, field
        r = {'mcc': {k: mcc(truth[I.region_slice(regions[k])], pred[I.region_slice(regions[k])])
                     for k in ('train', 'val', 'test')}}
        sl_val, sl_test = I.region_slice(regions['val']), I.region_slice(regions['test'])
        d_val = I.distance_to_tracer(vol.data, regions['val'], pad=PAD) * cell
        d_test = I.distance_to_tracer(vol.data, regions['test'], pad=PAD) * cell
        y_val, y_test, p_test = truth[sl_val], truth[sl_test], pred[sl_test]

        # 1. share of each true class's test voxels per distance bin
        which = np.clip(np.digitize(d_test, COARSE) - 1, 0, len(COARSE) - 2)
        r['class_share'] = {CLASS_NAMES[c]: (np.bincount(which[y_test == c], minlength=len(COARSE) - 1)
                                             / max((y_test == c).sum(), 1)).tolist() for c in range(4)}
        r['recall'] = {CLASS_NAMES[c]: I.binned_scores(d_test, p_test, y_test, COARSE)[f'recall_{c}'].tolist()
                       for c in range(4)}
        # 2. model against the distance-only classifier
        table = I.majority_by_distance(d_val, y_val, FINE)
        p_base = I.predict_by_distance(d_test, table, FINE)
        r['baseline_table'] = [CLASS_NAMES[c] for c in table]
        r['baseline_mcc'] = mcc(y_test, p_base)
        bm = I.binned_scores(d_test, p_test, y_test, FINE)
        bb = I.binned_scores(d_test, p_base, y_test, FINE)
        r['acc_model'], r['acc_base'], r['counts'] = bm['accuracy'].tolist(), bb['accuracy'].tolist(), bm['counts'].tolist()
        # 3. MCC if everything beyond D were right
        r['fixed_beyond'] = I.mcc_if_fixed_beyond(p_test, y_test, d_test, CUTOFFS)
        res[label][L] = r
        print(f"{label:15s} λ = {L:2d}: MCC train {r['mcc']['train']:.3f}  val {r['mcc']['val']:.3f}  "
              f"test {r['mcc']['test']:.3f} | distance-only {r['baseline_mcc']:.3f} | "
              f"test voxels > 8 h⁻¹ Mpc from a tracer: {(d_test > 8).mean():.1%}", flush=True)
        del pred, d_val, d_test, y_val, y_test, p_test, p_base, which
        gc.collect()
    del model, truth
    gc.collect()
rep.record(results=res, fine_edges_mpc=FINE, coarse_edges_mpc=COARSE[:-1], cutoffs_mpc=CUTOFFS,
           runs={k: v[0] for k, v in MODELS.items()})

## 1. What is far from every tracer

For each true class, the share of its test voxels in each distance bin (bars), and the model's recall in that bin (dots, right axis). A class with much of its volume beyond 6–8 h⁻¹ Mpc and low recall there is mostly out of the input's reach.

In [ ]:
colors = ['#9ca3af', '#6b7280', '#4b5563', '#1f2937', '#2563eb']
for label in MODELS:
    fig, axes = plt.subplots(1, len(SPACINGS), figsize=(5 * len(SPACINGS), 4.2), sharey=True)
    for ax, L in zip(axes, SPACINGS):
        r = res[label][L]
        x = np.arange(4)
        bottom = np.zeros(4)
        for b, bname in enumerate(COARSE_NAMES):
            share = np.array([r['class_share'][c][b] for c in CLASS_NAMES])
            ax.bar(x, share, 0.6, bottom=bottom, color=colors[b], label=bname + ' h⁻¹ Mpc')
            bottom += share
        ax2 = ax.twinx()
        far = [r['recall'][c][-1] for c in CLASS_NAMES]
        ax2.plot(x, far, 'o', color='#dc2626', label='recall beyond 8')
        ax2.set_ylim(0, 1)
        if L != SPACINGS[-1]:
            ax2.set_yticklabels([])
        else:
            ax2.set_ylabel('recall beyond 8 h⁻¹ Mpc (dots)')
        ax.set_xticks(x, CLASS_NAMES)
        ax.set_title(f'λ = {L}')
    axes[0].set_ylabel('share of the class\'s test voxels')
    handles, names = axes[0].get_legend_handles_labels()
    dot = plt.Line2D([], [], marker='o', color='#dc2626', ls='', label='recall beyond 8 h⁻¹ Mpc')
    fig.legend(handles + [dot], names + [dot.get_label()], loc='lower center', ncol=6, fontsize=9,
               title='distance to nearest tracer', bbox_to_anchor=(0.5, -0.12))
    fig.suptitle(f'{label}: how far each class is from the nearest tracer (test slab)')
    plt.tight_layout()
    rep.show(fig, f'1_class_distance_{label.replace(" ", "_")}')

## 2. Model against a distance-only classifier

Accuracy against distance to the nearest tracer, for the model (solid) and the classifier that only knows the distance (dashed). Grey: share of test voxels per bin. The gap is what the model gets from the input beyond distance: the shape and density of the tracers around it.

In [ ]:
mid = 0.5 * (FINE[1:] + FINE[:-1])
cmap = plt.get_cmap('viridis')
fig, axes = plt.subplots(1, len(MODELS), figsize=(7 * len(MODELS), 4.5), sharey=True)
for ax, label in zip(np.atleast_1d(axes), MODELS):
    for i, L in enumerate(SPACINGS):
        r = res[label][L]
        col = cmap(i / (len(SPACINGS) - 1))
        ax.plot(mid, r['acc_model'], '-', color=col, label=f'λ = {L}: model (MCC {r["mcc"]["test"]:.3f})')
        ax.plot(mid, r['acc_base'], '--', color=col, label=f'λ = {L}: distance only (MCC {r["baseline_mcc"]:.3f})')
    ax.set_title(label)
    ax.set_xlabel('distance to nearest tracer [h⁻¹ Mpc]')
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, ncol=2)
np.atleast_1d(axes)[0].set_ylabel('accuracy on the test slab')
fig.suptitle('Model against a classifier that sees only the distance to the nearest tracer')
plt.tight_layout()
rep.show(fig, '2_model_vs_distance_only')

## 3. How much score sits far from tracers

Four-class MCC on the test slab if every voxel farther than D from a tracer were classified correctly (solid), against the model's actual MCC (dotted). Where the solid line is still far above the dotted one at D = 8–10 h⁻¹ Mpc, much of what the model misses is in voxels with no tracer nearby.

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(7 * len(MODELS), 4.5), sharey=True)
for ax, label in zip(np.atleast_1d(axes), MODELS):
    for i, L in enumerate(SPACINGS):
        r = res[label][L]
        col = cmap(i / (len(SPACINGS) - 1))
        ax.plot(CUTOFFS, r['fixed_beyond'], '-o', color=col, label=f'λ = {L}')
        ax.axhline(r['mcc']['test'], color=col, ls=':', lw=1)
    ax.set_title(label)
    ax.set_xlabel('D: voxels farther than this from a tracer counted as correct [h⁻¹ Mpc]')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
np.atleast_1d(axes)[0].set_ylabel('four-class MCC, test slab')
fig.suptitle('MCC if every voxel beyond D were right (solid) and the actual MCC (dotted)')
plt.tight_layout()
rep.show(fig, '3_mcc_if_fixed_beyond')

## 4. Training, validation and test slabs

Four-class MCC of each model on the three slabs. The training slab was seen during training (as random crops); validation chose the saved model; the test slab was never used.

In [ ]:
for label in MODELS:
    print(label)
    print('  λ    train    val    test   train-test')
    for L in SPACINGS:
        m = res[label][L]['mcc']
        print(f"  {L:2d}   {m['train']:.3f}   {m['val']:.3f}   {m['test']:.3f}   {m['train'] - m['test']:+.3f}")

Everything above is saved in `deepvoid_runs/information_limit/`: one PNG per figure and `results.json` with the numbers.